In [ ]:
import os
import sys
import torch
import numpy as np
import torch.nn as nn
from pathlib import Path
import torchvision.datasets as Datasets
import pandas as pd
from pathlib import Path
import pickle
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import torch
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai import transforms
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from skimage.transform import rotate
from skimage.util import random_noise
import torch.nn.functional as F

pkg_path = str(Path(os.path.abspath('')).parent.absolute())
sys.path.insert(0, pkg_path)


data_path=pkg_path+'/results/'

from src import *

# Load config file
config = global_config.config
device = torch.device(config.device) 


2025-03-07 21:12:59.915925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741410779.927778 3997486 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741410779.931299 3997486 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 21:12:59.945436: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
direct_pairs='/home/ee577/project/results/DTI_AD_NC_paired_data.pkl'

with open(direct_pairs, 'rb') as f:
    X, y = pickle.load(f)

print(f"Loaded X shape: {X.shape}")
print(f"Loaded y shape: {y.shape}")


Loaded X shape: (499, 74, 98, 86)
Loaded y shape: (499, 55)


In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.7, random_state=42)

print(f"Training set: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Validation set: X_val={X_val.shape}, y_val={y_val.shape}")
print(f"Test set: X_test={X_test.shape}, y_test={y_test.shape}")

Training set: X_train=(399, 74, 98, 86), y_train=(399, 55)
Validation set: X_val=(30, 74, 98, 86), y_val=(30, 55)
Test set: X_test=(70, 74, 98, 86), y_test=(70, 55)


In [ ]:

def random_flip(image):
    """
    Randomly flips the image horizontally or vertically.
    
    Args:
    - image (numpy.ndarray or torch.Tensor): Input image, can be a numpy array or torch tensor.
    
    Returns:
    - torch.Tensor: Randomly flipped image.
    """
    if isinstance(image, np.ndarray):
        image = torch.tensor(image)
    
    if image.dim() == 3:
        image = image.unsqueeze(0)
    
    # (batch_size, channels, height, width)
    if image.dim() == 4:
        # Flip horizontally (along the width axis)
        if random.random() < 0.5:
            image = image.flip(3)  
        
        # Flip vertically (along the height axis)
        if random.random() < 0.5:
            image = image.flip(2)  
    else:
        raise ValueError(f"Expected image to have 3 or 4 dimensions, but got {image.dim()} dimensions.")
    
    # If we added a batch dimension earlier (for single image), remove it now
    if image.dim() == 4:
        image = image.squeeze(0)
    
    return image

def add_random_noise(image, noise_factor=0.05):
    """
    Adds random noise to the image.
    """
    noise = np.random.normal(0, noise_factor, image.shape)
    return image + noise


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        """
        Args:
            images (numpy array or torch tensor): 4D tensor with shape (N, D, H, W), where N is the number of samples
            labels (numpy array or torch tensor): 2D tensor with shape (N, num_features), where N is the number of samples
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if isinstance(image, np.ndarray):
            image = torch.tensor(image, dtype=torch.float32)

        label = torch.tensor(label, dtype=torch.float32)
        
        if image.ndimension() == 3:  # Shape: (D, H, W)
            image = image.unsqueeze(0)  # Add the channel dimension (Shape becomes: (1, D, H, W))
        
        if self.transform:
            image = self.transform(image)
        
        return {'image': image, 'label': label}

In [ ]:
class UNet3DRegression(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet3DRegression, self).__init__()
        self.unet = UNet(
            spatial_dims=3,  # Use 3D convolution
            in_channels=in_channels,  # Input channels
            out_channels=out_channels,  # Output channels
            channels=(16, 32, 64, 128),  # Number of channels in each layer
            strides=(2, 2, 2),  # Strides for downsampling
            kernel_size=3,  # Kernel size for convolutions
            up_kernel_size=3,  # Kernel size for upsampling
            num_res_units=2,  # Number of residual units
            act='PReLU',  # Activation function
            norm='INSTANCE',  # Normalization method
            dropout=0.1,  # Dropout rate
        )

        # will be adjusted in the forward pass
        self.regression_layer = None

    def forward(self, x):
        # Check input size and resize if necessary
        _, _, depth, height, width = x.size()

        # Ensure the input size is divisible by 16 (since we are using 3 downsampling layers with strides of 2)
        if depth % 16 != 0 or height % 16 != 0 or width % 16 != 0:
            new_depth = (depth // 16 + 1) * 16
            new_height = (height // 16 + 1) * 16
            new_width = (width // 16 + 1) * 16
            x = F.interpolate(x, size=(new_depth, new_height, new_width), mode='trilinear', align_corners=True)

        # Apply the U-Net forward pass
        x = self.unet(x)

        # Get the flattened size dynamically (size[1:] are the output channels, depth, height, width)
        flattened_size = x.numel() // x.size(0)  

        if self.regression_layer is None:
            self.regression_layer = nn.Linear(flattened_size, 55)
        x = x.view(x.size(0), -1)  # Flattens the output to (batch_size, features)
        x = self.regression_layer(x)

        return x


In [7]:
def save_checkpoint(model, optimizer, epoch, loss, path='/home/ee577/project/results/checkpoint.pth'):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path}")


In [ ]:
def train_model(model, train_loader, optimizer, loss_function, num_epochs=1000, seeds=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]):
    """
    Train the model with random transformations applied to each batch of images and iterate over multiple seeds.

    Args:
        model: The neural network model.
        train_loader: DataLoader containing the training dataset.
        optimizer: The optimizer for training.
        loss_function: The loss function (e.g., MSE for regression).
        num_epochs (int, optional): Number of epochs to train. Defaults to 1000.
        seeds (list, optional): List of random seeds to use. Defaults to [1, 2, 3, ..., 10].
    """
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        # Select the seed for this epoch from the list
        seed = seeds[epoch % len(seeds)]  
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        running_loss = 0.0

        # Iterate over batches in the train_loader
        for batch in train_loader:
            inputs, labels = batch['image'], batch['label']
            inputs = inputs.to(device)
            labels = labels.to(device)

            transformed_inputs = []
            for i in range(inputs.size(0)):  # Loop through each image in the batch
                image = inputs[i].cpu().numpy()  # Convert to numpy for transformation

                # Apply random transformations (flip, rotate, noise)
                if random.random() < 0.5:
                    image = random_flip(image)
                if random.random() < 0.5:
                    image = add_random_noise(image)

                transformed_inputs.append(image.clone().detach().to(dtype=torch.float32))  

            transformed_inputs = torch.stack(transformed_inputs).to(device)  
            if len(transformed_inputs.shape) == 4:  # (batch_size, channels, height, width)
                transformed_inputs = transformed_inputs.unsqueeze(1) 
            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(transformed_inputs)

            # Calculate loss
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        torch.cuda.empty_cache()

        # Average loss for this epoch
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Seed {seed}, Loss: {avg_loss:.4f}")

        # Delete variables to free memory
        del inputs, labels, transformed_inputs
        torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")



In [ ]:
train_loader=CustomDataset(X_train, y_train)
for batch in train_loader:
    in_shape = batch['label'].shape  
    break  
print(in_shape)


torch.Size([55])


In [15]:
train_loader=CustomDataset(X_train, y_train)
val_loader=CustomDataset(X_val, y_val)
in_shape = batch['image'].shape 
out_shape=y_train.shape[1:][0]
print(in_shape)
model = UNet3DRegression(in_channels=1, out_channels=out_shape)

print(model)
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


seeds = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # The list of seeds to iterate over
train_model(model, train_loader, optimizer, loss_function, num_epochs=100, seeds=seeds).to(device)

torch.Size([1, 74, 98, 86])
UNet3DRegression(
  (unet): UNet(
    (model): Sequential(
      (0): ResidualUnit(
        (conv): Sequential(
          (unit0): Convolution(
            (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
            (adn): ADN(
              (N): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
              (D): Dropout(p=0.1, inplace=False)
              (A): PReLU(num_parameters=1)
            )
          )
          (unit1): Convolution(
            (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (adn): ADN(
              (N): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
              (D): Dropout(p=0.1, inplace=False)
              (A): PReLU(num_parameters=1)
            )
          )
        )
        (residual): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
      )
   

/tmp/ipykernel_3997486/135860179.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  transformed_inputs.append(torch.tensor(image, dtype=torch.float32))  # Ensure correct dtype


KeyboardInterrupt: 

In [ ]:
def create_3d_cnn_with_regression(output_size):
    # Define the input layer with shape (depth, height, width, channels)
    input_layer = layers.Input(shape=(10, 64, 64, 1))  # (depth, height, width, channels)

    # 3D convolutional layers
    x = layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same')(input_layer)
    x = layers.MaxPooling3D((2, 2, 2))(x)  # Pooling over the 3D space

    x = layers.Conv3D(64, (3, 3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    x = layers.Conv3D(128, (3, 3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    # Flatten the output for the dense layer
    x = layers.Flatten()(x)

    # Dense layer (optional, can be adjusted as needed)
    x = layers.Dense(256, activation='relu')(x)

    # Output layer for regression (e.g., predicting continuous values)
    output_layer = layers.Dense(output_size)(x)  # output_size can vary (e.g., 1 for a single output, or more)

    # Create and compile the model
    model = models.Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

    return model


